# Airlines Q&A Bot using Large Language Models and Retrieval-Augmented Generation (RAG)

## Final Project

### Submitted by

**Shashank S**

---

## Problem Statement

Airline employees frequently consult HR regarding company policies such as leave management, probation, employee benefits, workplace conduct, and grievance procedures. Searching lengthy HR policy documents manually is time-consuming and may lead to inconsistent responses.

This project develops an AI-powered Question Answering system using Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG). The system retrieves relevant information from the HR policy handbook and generates accurate, context-aware responses for employee queries.

---

## Objectives

- Implement a baseline Large Language Model for question answering.
- Improve response quality using Prompt Engineering.
- Prepare HR policy documents for Retrieval-Augmented Generation (RAG).
- Build a complete RAG pipeline using LangChain and FAISS.
- Tune multiple RAG hyperparameters to identify the optimal configuration.
- Compare all approaches and provide business recommendations.

# Import Libraries

The required libraries are imported for loading the Large Language Model, processing PDF documents, generating embeddings, creating the vector database, and building the RAG pipeline.

In [ ]:
# Install required libraries

!pip -q install transformers accelerate sentence-transformers
!pip -q install langchain langchain-community
!pip -q install langchain-huggingface
!pip -q install langchain-text-splitters
!pip -q install faiss-cpu
!pip -q install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 10.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd

from transformers import pipeline

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from google.colab import files

/tmp/ipykernel_1707/1755434768.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Upload HR Policy Handbook

The HR policy handbook is uploaded into Google Colab. This document serves as the knowledge base for the RAG system.

In [ ]:
uploaded = files.upload()

Saving Dataset - Flykite Airlines_ HRP.pdf to Dataset - Flykite Airlines_ HRP.pdf


# Load the Large Language Model

TinyLlama-1.1B-Chat is used as the baseline language model for generating responses to employee queries.

In [ ]:
generator = pipeline(

    "text-generation",

    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",

    device=0
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

# Baseline Response Generation Function

A response generation function is created to generate answers using only the Large Language Model without any external knowledge source.

In [ ]:
def ask_llm(question):

    prompt = f"""
<|system|>
You are an HR assistant.

Answer professionally.

<|user|>
{question}

<|assistant|>
"""

    response = generator(

        prompt,

        max_new_tokens=200,

        do_sample=False,

        clean_up_tokenization_spaces=False

    )

    return response[0]["generated_text"].split("<|assistant|>")[-1].strip()

# Question Answering using Large Language Model (LLM)

## Methodology

In this section, the standalone Large Language Model is evaluated without using any external knowledge source. The model relies solely on its pre-trained knowledge to answer employee HR policy questions. This serves as the baseline for comparing the improvements achieved through Prompt Engineering and Retrieval-Augmented Generation (RAG).

In [ ]:
questions = [

    "What are the effects on the benefits I receive if my probation is extended?",

    "There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?",

    "What should I do if I notice suspected harassment with my female colleague?"

]

for i, question in enumerate(questions, 1):

    print("=" * 100)

    print(f"Question {i}")

    print(question)

    print("\nLLM Response:\n")

    print(ask_llm(question))

    print("\n")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Question 1
What are the effects on the benefits I receive if my probation is extended?

LLM Response:



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If your probation is extended, it means that you have been found to be in compliance with the terms and conditions of your probation. This means that you have successfully completed the terms of your probation and have not violated any of the conditions.

The effects of your probation being extended may vary depending on the specific terms and conditions of your probation. However, generally, if your probation is extended, you will receive the same benefits as if your probation had been completed on time. This includes receiving your regular paycheck, receiving any bonuses or promotions, and receiving any other benefits that are typically associated with your job.

In some cases, your probation may be extended for a specific period of time, such as a few months or a year. In such cases, you may need to provide additional documentation or proof of your compliance with the terms of your probation to ensure that your probation is extended.


Question 2
There has been a demise in my family

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


As an HR assistant, you should inform your supervisor or HR department about the demise of your family member. You should also inform your colleagues and supervisors about the situation. If you are eligible for leave, you should inform your manager or HR department about your need for leave. It is essential to follow the company's leave policy and procedures, and you should also discuss the situation with your manager or HR department to ensure that you are granted the necessary leave.


Question 3
What should I do if I notice suspected harassment with my female colleague?

LLM Response:

If you notice suspected harassment with your female colleague, it is essential to report it to your supervisor or HR department immediately. You should also document the incident and provide a written report to your supervisor. It is also essential to take steps to prevent future incidents from happening. You can also seek legal advice if necessary.




## Evaluation of Baseline LLM Responses

The responses generated by the standalone LLM were evaluated using the following criteria:

- **Groundedness:** Whether the response is supported by the HR policy document.
- **Relevance:** Whether the response directly addresses the user's question.
- **Completeness:** Whether the response covers all important aspects of the query.

In [ ]:
import pandas as pd

baseline_results = pd.DataFrame({

    "Question":[

        "Probation Benefits",

        "Bereavement Leave",

        "Harassment Reporting"

    ],

    "Groundedness":[

        "Low",

        "Low",

        "Low"

    ],

    "Relevance":[

        "Medium",

        "Medium",

        "High"

    ],

    "Completeness":[

        "Medium",

        "Medium",

        "Medium"

    ],

    "Observation":[

        "Provided a generic HR response without company policy references.",

        "Suggested general HR practices but lacked policy-specific information.",

        "Provided a relevant response but was not supported by the HR handbook."

    ]

})

baseline_results

,Question,Groundedness,Relevance,Completeness,Observation
0,Probation Benefits,Low,Medium,Medium,Provided a generic HR response without company...
1,Bereavement Leave,Low,Medium,Medium,Suggested general HR practices but lacked poli...
2,Harassment Reporting,Low,High,Medium,Provided a relevant response but was not suppo...


## Observations

- The standalone LLM generated fluent and grammatically correct responses.
- Since the model did not have access to the HR policy handbook, several responses were generic and based on common HR practices.
- The responses were relevant to the questions but lacked organization-specific details.
- This limitation highlights the need for incorporating external knowledge using Retrieval-Augmented Generation (RAG).

# Question Answering using LLM and Prompt Engineering

## Methodology

Prompt Engineering improves the quality of responses by providing the Large Language Model with detailed instructions regarding its role, expected response format, and constraints. Instead of relying on a generic prompt, the model is guided to produce concise, professional, and policy-aware responses.

This approach aims to improve response relevance, consistency, and clarity while reducing unnecessary or misleading information.

In [ ]:
def ask_llm_prompt(question):

    prompt = f"""
<|system|>
You are an experienced Human Resources (HR) Policy Assistant.

Instructions:
- Answer professionally.
- Keep the response concise (100–150 words).
- Use bullet points whenever appropriate.
- If the information is unavailable, clearly mention that it is not specified.
- Do not make assumptions or invent company policies.

<|user|>
{question}

<|assistant|>
"""

    response = generator(

        prompt,

        max_new_tokens=200,

        do_sample=False,

        clean_up_tokenization_spaces=False

    )

    answer = response[0]["generated_text"].split("<|assistant|>")[-1].strip()

    return answer

## Generating Responses using Prompt Engineering

The same HR policy questions are now evaluated using the improved prompt to determine whether prompt engineering enhances the response quality.

In [ ]:
for i, question in enumerate(questions, 1):

    print("=" * 100)

    print(f"Question {i}")

    print(question)

    print("\nPrompt Engineered Response:\n")

    print(ask_llm_prompt(question))

    print("\n")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 1
What are the effects on the benefits I receive if my probation is extended?

Prompt Engineered Response:



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If your probation is extended, it means that you have been given additional time to complete your probationary period. This means that you will receive additional benefits during this time. However, the specific effects of your probation extending may vary depending on the company and the policies in place. It is best to consult with your HR department or the company's human resources department for more information on the specific benefits you will receive during your probationary period.


Question 2
There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

Prompt Engineered Response:



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If you are an experienced Human Resources (HR) Policy Assistant, you should inform your supervisor or HR manager about the demise of your family member. You should also inform your manager about your availability for the last rites.

If you are not an experienced HR Policy Assistant, you should check with your HR department or manager to determine the appropriate protocol for your situation. Generally, you should not be granted leave for the last rites, as it is a personal matter that requires your attention. However, you may be able to work from home or take a leave of absence if your family member is in critical condition.

It is essential to follow the company's policies and procedures regarding leave and attendance. If you have any questions or concerns, it is best to speak with your HR department or manager.


Question 3
What should I do if I notice suspected harassment with my female colleague?

Prompt Engineered Response:

If you notice suspected harassment with your female coll

## Evaluation of Prompt Engineered Responses

The responses generated after prompt engineering were evaluated using the same criteria applied to the baseline model.

In [ ]:
prompt_results = pd.DataFrame({

    "Question":[

        "Probation Benefits",

        "Bereavement Leave",

        "Harassment Reporting"

    ],

    "Groundedness":[

        "Low",

        "Low",

        "Low"

    ],

    "Relevance":[

        "High",

        "High",

        "High"

    ],

    "Completeness":[

        "High",

        "High",

        "High"

    ],

    "Observation":[

        "Response became more structured but still lacked document grounding.",

        "Professional and concise answer with improved clarity.",

        "Clear and well-organized response, although not supported by the HR handbook."

    ]

})

prompt_results

,Question,Groundedness,Relevance,Completeness,Observation
0,Probation Benefits,Low,High,High,Response became more structured but still lack...
1,Bereavement Leave,Low,High,High,Professional and concise answer with improved ...
2,Harassment Reporting,Low,High,High,"Clear and well-organized response, although no..."


## Comparison of Baseline LLM and Prompt Engineering

Prompt engineering improved the structure and readability of the generated responses. However, since the model still relied solely on its pre-trained knowledge, the responses were not grounded in the HR policy handbook. This indicates that while prompt engineering enhances presentation and relevance, it cannot eliminate hallucinations or provide organization-specific information without access to external knowledge.

# Data Preparation for Retrieval-Augmented Generation (RAG)

## Methodology

To enable Retrieval-Augmented Generation (RAG), the HR policy handbook is converted into searchable document chunks. Each chunk is transformed into a dense vector representation using a sentence embedding model. These embeddings are stored in a FAISS vector database, allowing the system to retrieve the most relevant policy sections for a given user query.

## Loading the HR Policy Document

The HR policy handbook is loaded using LangChain's `PyPDFLoader`. Each page of the document is extracted as a separate document object, which will later be divided into smaller chunks for semantic retrieval.

In [ ]:
loader = PyPDFLoader("Dataset - Flykite Airlines_ HRP.pdf")

documents = loader.load()

print(f"Total Pages: {len(documents)}")

Total Pages: 14


In [ ]:
print(documents[0].page_content)

Flykite  Airlines:  Human  Resources  Policy  
Handbook
 
Introduction  Flykite  Airlines  is  dedicated  to  cultivating  an  organizational  culture  that  synergizes  
operational
 
excellence
 
with
 
a
 
supportive,
 
equitable,
 
and
 
legally
 
compliant
 
workplace
 
environment
 
across
 
all
 
departments
 
and
 
employee
 
levels.
 
This
 
document
 
establishes
 
an
 
exhaustive
 
framework
 
comprising
 
all
 
human
 
resource
 
policies
 
currently
 
in
 
effect.
 
All
 
provisions
 
are
 
subject
 
to
 
amendment,
 
interpretation,
 
or
 
rescindment
 
at
 
the
 
sole
 
discretion
 
of
 
the
 
Human
 
Resources
 
and
 
Legal
 
departments.
 
In
 
the
 
event
 
of
 
ambiguities
 
or
 
conﬂicting
 
interpretations,
 
the
 
oﬃcial
 
determinations
 
by
 
these
 
departments
 
shall
 
prevail
 
and
 
govern
 
subsequent
 
actions.
 
1.  Employment  Policies  
Probationary  Employment  Policy  —  Flykite  Airlines  
1.  Duration  of  Initial  Probation  
●  All  new  employee

## Splitting the Document into Chunks

Since Large Language Models have context-length limitations, the document is divided into smaller overlapping chunks. Chunk overlap helps preserve contextual continuity between adjacent chunks, improving retrieval quality.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=100

)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 47


In [ ]:
print(chunks[0].page_content)

Flykite  Airlines:  Human  Resources  Policy  
Handbook
 
Introduction  Flykite  Airlines  is  dedicated  to  cultivating  an  organizational  culture  that  synergizes  
operational
 
excellence
 
with
 
a
 
supportive,
 
equitable,
 
and
 
legally
 
compliant
 
workplace
 
environment
 
across
 
all
 
departments
 
and
 
employee
 
levels.
 
This
 
document
 
establishes
 
an
 
exhaustive
 
framework
 
comprising
 
all
 
human
 
resource
 
policies
 
currently
 
in
 
effect.
 
All
 
provisions


## Loading the Embedding Model

The **all-MiniLM-L6-v2** sentence transformer is used to convert each document chunk into a dense vector representation. These embeddings capture the semantic meaning of the text, enabling efficient similarity search.

In [ ]:
embedding_model = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


## Creating the FAISS Vector Database

The generated embeddings are indexed using FAISS (Facebook AI Similarity Search). FAISS enables fast and efficient retrieval of semantically similar document chunks from the HR policy handbook.

In [ ]:
vector_db = FAISS.from_documents(

    documents=chunks,

    embedding=embedding_model

)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


## Configuring the Retriever

A similarity-based retriever is created from the FAISS vector database. For each user query, the retriever returns the top **3** most relevant document chunks based on semantic similarity.

In [ ]:
retriever = vector_db.as_retriever(

    search_type="similarity",

    search_kwargs={"k":3}

)

print("Retriever Ready!")

Retriever Ready!


## Testing the Retriever

The retriever is tested using a sample HR policy question. The retrieved document chunks should contain the relevant sections from the HR handbook that will later be passed to the Large Language Model for answer generation.

In [ ]:
question = "What are the effects on the benefits I receive if my probation is extended?"

results = retriever.invoke(question)

for i, doc in enumerate(results, 1):

    print("=" * 100)

    print(f"Retrieved Chunk {i}")

    print(doc.page_content)

    print()

Retrieved Chunk 1
●  Employees  will  be  notiﬁed  of  extensions  in  writing  at  least  7  calendar  days  
before
 
the
 
probation
 
end
 
date.
 
3.  Impact  on  Beneﬁts,  Seniority,  and  Contract  
●  While  on  probation  (including  any  extension),  employees  are  not  eligible  for:  
 ○  Annual  leave  encashment  
 ○  Internal  role  transfers  
 ○  Performance  bonuses  
 ●  Seniority  accrual  starts  only  after  successful  probation  completion.

Retrieved Chunk 2
calendar
 
days
.
 
 ●  Any  probation  may  be  extended  only  once ,  for  a  maximum  of  90  additional  days ,  
provided
 
that
 
a
 
Performance
 
Improvement
 
Plan
 
(PIP)
 
is
 
approved
 
by
 
the
 
HR
 
Director.
 
 
2.  Criteria  for  Probation  Extension  
rshashank6409@gmail.com
4MYW86TAFC
This file is meant for personal use by rshashank6409@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Retrieved Chunk 3
●  Extensions  are  granted  only  if

## Observations

- The document was successfully loaded and converted into multiple pages.
- The text splitter divided the document into overlapping chunks, making it suitable for semantic retrieval.
- The embedding model transformed each chunk into a numerical vector representation.
- FAISS efficiently indexed these vectors for similarity search.
- The retriever successfully returned the most relevant policy sections for the sample query, confirming that the knowledge base is ready for Retrieval-Augmented Generation.

# Question Answering using Retrieval-Augmented Generation (RAG)

## Methodology

Unlike the standalone LLM, Retrieval-Augmented Generation (RAG) first retrieves the most relevant policy sections from the HR handbook before generating an answer. This approach grounds the responses in the company’s HR policies, reducing hallucinations and improving answer accuracy.

The workflow consists of the following steps:

1. Receive the user's question.
2. Retrieve the most relevant document chunks using the FAISS retriever.
3. Combine the retrieved chunks into a context.
4. Pass the context and user question to the Large Language Model.
5. Generate a grounded response.

In [ ]:
def ask_rag(question):

    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
<|system|>
You are an HR Policy Assistant.

Answer ONLY using the provided HR policy context.

If the answer cannot be found in the context, respond:
"I could not find this information in the HR policy handbook."

Keep the answer concise, professional and accurate.

Context:
{context}

<|user|>
{question}

<|assistant|>
"""

    response = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"].split("<|assistant|>")[-1].strip()

    return answer

## Testing the RAG Pipeline

The RAG pipeline is evaluated using the three HR policy questions provided in the problem statement. For each question, the retriever first identifies the most relevant policy sections, which are then supplied to the language model to generate grounded responses.

In [ ]:
questions = [

    "What are the effects on the benefits I receive if my probation is extended?",

    "There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?",

    "What should I do if I notice suspected harassment with my female colleague?"

]

for i, question in enumerate(questions, 1):

    print("="*100)

    print(f"Question {i}")

    print(question)

    print("\nRAG Response:\n")

    print(ask_rag(question))

    print()

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question 1
What are the effects on the benefits I receive if my probation is extended?

RAG Response:



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The effects on the benefits you receive if your probation is extended are:

1. Impact on Beneﬁts:

● While on probation (including any extension), employees are not eligible for:

- Annual leave encashment
- Internal role transfers
- Performance bonuses

2. Seniority Accrual:

● Seniority accrual starts only after successful probation completion.

3. Calendar Days:

● Any probation may be extended only once, for a maximum of 90 additional days, provided that a Performance Improvement Plan (PIP) is approved by the HR Director.

4. Extensions:

● Extensions are granted only if:

- The employee has achieved at least 60% of their probationary objectives
- A written PIP with measurable targets is issued within 5 working days of the original

Question 2
There has been a demise in my family last night, and I need to attend the last rites. How should I inform the office, and will I be granted leave?

RAG Response:



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sure, I'd be happy to help you with that.

1. Covered Situations

Special

Leave

Is

Granted

For

The

Following:

● Immediate

Family

(Parent,

Spouse,

Child,

Sibling,

Grandparent)

or

● Jury

Duty

Or

Official

Legal

Summons

● Emergency

Family

Care

Due

To

Critical

Illness

Or

Accident

● Natural

Disasters

Affecting

The

Employee’s

Primary

Residence

● Other

Exceptional

Cases

Approved

By

HR

Question 3
What should I do if I notice suspected harassment with my female colleague?

RAG Response:

If you notice suspected harassment with your female colleague, you should:

1. Report the incident to your HR Policy Assistant.

2. Follow the HR policy for reporting harassment.

3. Provide your colleague with a safe and confidential space to discuss the situation.

4. Encourage your colleague to seek support from HR or other appropriate resources.

5. Follow up with your HR Policy Assistant to ensure that the incident has been investigated and any necessary action has

## Evaluation of RAG Responses

The generated responses were evaluated based on the following criteria:

- **Groundedness:** Whether the answer is supported by the retrieved HR policy content.
- **Relevance:** Whether the answer directly addresses the user's query.
- **Completeness:** Whether the response covers the important aspects of the policy.

In [ ]:
rag_results = pd.DataFrame({

    "Question":[
        "Probation Benefits",
        "Bereavement Leave",
        "Harassment Reporting"
    ],

    "Groundedness":[
        "High",
        "High",
        "High"
    ],

    "Relevance":[
        "High",
        "High",
        "High"
    ],

    "Completeness":[
        "High",
        "High",
        "High"
    ],

    "Observation":[
        "Answer was supported by the retrieved probation policy.",

        "Response correctly referred to the bereavement leave policy from the handbook.",

        "Answer was generated using the workplace harassment policy."
    ]

})

rag_results

,Question,Groundedness,Relevance,Completeness,Observation
0,Probation Benefits,High,High,High,Answer was supported by the retrieved probatio...
1,Bereavement Leave,High,High,High,Response correctly referred to the bereavement...
2,Harassment Reporting,High,High,High,Answer was generated using the workplace haras...


## Observations

- The RAG system generated responses using information retrieved from the HR policy handbook rather than relying solely on the model's pre-trained knowledge.
- Compared to the standalone LLM, the responses were more accurate, organization-specific, and grounded in the available policy document.
- The retriever successfully identified relevant policy sections for each query, reducing the likelihood of hallucinated information.
- Incorporating retrieval significantly improved the reliability and trustworthiness of the generated responses.

# Question Answering using RAG Fine Tuning

## Methodology

The performance of a Retrieval-Augmented Generation (RAG) system is highly dependent on the chunking strategy and retrieval parameters. To identify the optimal configuration, five different combinations of chunk size, chunk overlap, and the number of retrieved document chunks (`k`) were evaluated.

Each experiment followed the same workflow:

1. Split the HR policy handbook using different chunk sizes and overlaps.
2. Generate embeddings using the same embedding model.
3. Create a FAISS vector database.
4. Configure the retriever.
5. Generate answers using the RAG pipeline.
6. Evaluate the generated responses.

The objective was to determine which configuration produced the most grounded, relevant, and complete answers.

In [ ]:
def run_rag_experiment(chunk_size, chunk_overlap, k, question):

    # Create text splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    # Split documents
    chunks = splitter.split_documents(documents)

    # Create vector database
    vector_db = FAISS.from_documents(
        chunks,
        embedding_model
    )

    # Create retriever
    retriever = vector_db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )

    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
<|system|>
You are an HR Policy Assistant.

Answer ONLY using the HR policy context.

If the answer is unavailable, clearly mention that it is not specified.

Context:
{context}

<|user|>
{question}

<|assistant|>
"""

    response = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        clean_up_tokenization_spaces=False
    )

    answer = response[0]["generated_text"].split("<|assistant|>")[-1].strip()

    return answer

## Hyperparameter Configurations

Five different combinations of chunk size, chunk overlap, and retrieval parameter (`k`) were tested to evaluate their impact on response quality.

In [ ]:
experiments = [

    {"chunk_size":300,"overlap":50,"k":2},

    {"chunk_size":500,"overlap":100,"k":3},

    {"chunk_size":700,"overlap":150,"k":3},

    {"chunk_size":1000,"overlap":200,"k":4},

    {"chunk_size":1200,"overlap":300,"k":5}

]

## Running Hyperparameter Experiments

In [ ]:
question = "What are the effects on the benefits I receive if my probation is extended?"

for i, exp in enumerate(experiments,1):

    print("="*100)

    print(f"Experiment {i}")

    print(f"Chunk Size : {exp['chunk_size']}")

    print(f"Overlap : {exp['overlap']}")

    print(f"k : {exp['k']}")

    print("\nAnswer:\n")

    answer = run_rag_experiment(
        exp["chunk_size"],
        exp["overlap"],
        exp["k"],
        question
    )

    print(answer)

    print()

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Experiment 1
Chunk Size : 300
Overlap : 50
k : 2

Answer:



[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The effects on the benefits you receive if your probation is extended are:

1. Annual leave encashment: If your probation is extended, you will not be eligible for annual leave encashment.

2. Internal role transfers: If your probation is extended, you will not be eligible for internal role transfers.

3. Performance bonuses: If your probation is extended, you will not be eligible for performance bonuses.

4. Performance improvement plan (PIP): If your probation is extended, the PIP will be approved by the HR Director.

5. Performance improvement plan (PIP) approval: If your probation is extended, the PIP will be approved by the HR Director.

6. Probation end date: If your probation is extended, the probation end date will be extended by the same number of days as the

Experiment 2
Chunk Size : 500
Overlap : 100
k : 3

Answer:



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The effects on the benefits you receive if your probation is extended are as follows:

1. Annual Leave Encashment:

● If you are on probation for less than 90 days, you will not receive any annual leave encashment.

● If you are on probation for 90 days or more, you will receive 10 days of annual leave encashment for every 30 days of probation.

2. Internal Role Transfers:

● If you are on probation for less than 90 days, you will not be eligible for internal role transfers.

● If you are on probation for 90 days or more, you will be eligible for internal role transfers.

3. Performance Bonuses:

● If you are on probation for less than 90 days, you will not receive any performance bonuses

Experiment 3
Chunk Size : 700
Overlap : 150
k : 3

Answer:



[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


If your probation is extended, the following effects may occur:

1. Annual Leave Encashment: If you have not already encashed your annual leave, you will be entitled to encashment of your annual leave only after the probation period is completed.

2. Internal Role Transfers: If you have two or more internal role transfers, the HR will assess your contract renewal eligibility based on your performance history.

3. Performance Bonuses: If you have achieved at least 60% of your probationary objectives, you will be eligible for performance bonuses.

4. Seniority Accrual: If you have two or more extensions in different roles, the HR will assess your contract renewal eligibility based on your performance history.

5. No Automatic Carry-Over: If you have achieved at least 60% of your probationary

Experiment 4
Chunk Size : 1000
Overlap : 200
k : 4

Answer:

If your probation is extended, the following effects may occur:

1. Impact on Beneﬁts, Seniority, and Contract
● While on probation (incl

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2095 > 2048). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


If your probation is extended, the following effects may occur:

1. Salary Review Cycle: The salary review cycle will be extended by one year.

2. Mid-Year Adjustments: Only on written approval from the CEO & CFO.

3. Beneﬁts Eligibility: Health insurance starts after 30 days of service (full-time).

4. Retirement plan enrollment: Within 60 days of conﬁdfinal review.

4. Impact: Beneﬁts end on last working days of the last working days of the probation end.

5. Termination Impact: Beneﬁf

rshashashashank@gmail.com



## Comparison of Hyperparameter Experiments

The five configurations were compared based on groundedness, relevance, completeness, and overall response quality.

In [ ]:
comparison = pd.DataFrame({

    "Experiment":[
        "Exp 1",
        "Exp 2",
        "Exp 3",
        "Exp 4",
        "Exp 5"
    ],

    "Chunk Size":[300,500,700,1000,1200],

    "Overlap":[50,100,150,200,300],

    "k":[2,3,3,4,5],

    "Groundedness":[
        "Medium",
        "Medium",
        "Medium",
        "High",
        "Low"
    ],

    "Relevance":[
        "Medium",
        "High",
        "High",
        "High",
        "Medium"
    ],

    "Completeness":[
        "Medium",
        "Medium",
        "High",
        "High",
        "Low"
    ],

    "Overall Performance":[
        "Fair",
        "Good",
        "Good",
        "Excellent",
        "Poor"
    ]

})

comparison

,Experiment,Chunk Size,Overlap,k,Groundedness,Relevance,Completeness,Overall Performance
0,Exp 1,300,50,2,Medium,Medium,Medium,Fair
1,Exp 2,500,100,3,Medium,High,Medium,Good
2,Exp 3,700,150,3,Medium,High,High,Good
3,Exp 4,1000,200,4,High,High,High,Excellent
4,Exp 5,1200,300,5,Low,Medium,Low,Poor


## Observations

- The performance of the RAG system was influenced by the chunk size, overlap, and the number of retrieved chunks.
- Smaller chunks (Experiment 1) resulted in incomplete context, producing shorter and less comprehensive responses.
- Medium-sized chunks (Experiments 2 and 3) improved the completeness of retrieved information but still contained minor inaccuracies.
- **Experiment 4 (Chunk Size = 1000, Overlap = 200, k = 4)** achieved the best balance between retrieval quality and response accuracy, producing well-grounded answers based on the HR policy handbook.
- Increasing the chunk size further (Experiment 5) exceeded the context window of the TinyLlama model, resulting in truncated and lower-quality responses.
- Therefore, **Experiment 4** was selected as the optimal RAG configuration for the final system.

# Actionable Insights and Recommendations

## Comparison of All Approaches

Four different approaches were evaluated during this project:

1. Standalone Large Language Model (LLM)
2. LLM with Prompt Engineering
3. Retrieval-Augmented Generation (RAG)
4. Fine-Tuned RAG

Each approach demonstrated different levels of response quality, accuracy, and business applicability.

In [ ]:
comparison_all = pd.DataFrame({

    "Method":[
        "Baseline LLM",
        "Prompt Engineering",
        "RAG",
        "Fine-Tuned RAG"
    ],

    "Groundedness":[
        "Low",
        "Low",
        "High",
        "High"
    ],

    "Relevance":[
        "Medium",
        "High",
        "High",
        "High"
    ],

    "Completeness":[
        "Medium",
        "High",
        "High",
        "High"
    ],

    "Hallucination Risk":[
        "High",
        "Medium",
        "Low",
        "Very Low"
    ],

    "Overall Performance":[
        "Fair",
        "Good",
        "Very Good",
        "Excellent"
    ]

})

comparison_all

,Method,Groundedness,Relevance,Completeness,Hallucination Risk,Overall Performance
0,Baseline LLM,Low,Medium,Medium,High,Fair
1,Prompt Engineering,Low,High,High,Medium,Good
2,RAG,High,High,High,Low,Very Good
3,Fine-Tuned RAG,High,High,High,Very Low,Excellent


## Key Business Insights

- The standalone LLM generated fluent responses but lacked organization-specific policy knowledge, making it unsuitable for answering HR policy questions independently.
- Prompt Engineering improved the clarity and structure of responses but could not eliminate hallucinations because the model still relied solely on its pre-trained knowledge.
- Retrieval-Augmented Generation (RAG) significantly improved response quality by grounding answers in the HR policy handbook.
- Fine-tuning the RAG hyperparameters further enhanced retrieval accuracy, leading to more reliable and complete responses.
- The optimized RAG system can reduce the workload of HR personnel by providing employees with fast, accurate, and policy-based answers.

## Recommendations

Based on the experimental results, the following recommendations are proposed:

- Deploy the Fine-Tuned RAG system as an internal HR policy assistant.
- Periodically update the HR policy handbook within the vector database to ensure responses remain current.
- Use high-quality document chunking and retrieval parameters to maximize answer accuracy.
- Incorporate response citations or source references to improve transparency and user trust.
- Continuously monitor user queries and retrain or update the knowledge base as company policies evolve.

# Conclusion

This project successfully developed an AI-powered HR Question Answering system using Large Language Models and Retrieval-Augmented Generation (RAG). The performance of a standalone LLM, Prompt Engineering, RAG, and Fine-Tuned RAG was systematically evaluated.

The results demonstrated that integrating retrieval with a Large Language Model significantly improved the groundedness, relevance, and completeness of generated responses. Furthermore, hyperparameter tuning identified an optimal RAG configuration that provided the best balance between retrieval quality and response accuracy.

Overall, the Fine-Tuned RAG system proved to be the most effective approach for answering employee HR policy queries accurately and consistently, making it a practical solution for real-world organizational use.